<a href="https://colab.research.google.com/github/marcohinojosag/IA_MKTR/blob/branch-y-histograma/cod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Importación de librerias y carga del dataset
Quien necesita descargar en 2025, las APIs son el futuro 🗣🗣🗣

In [9]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
import pandas as pd
import numpy as np
from kagglehub import KaggleDatasetAdapter
import re, numpy as np, pandas as pd
from scipy.stats import skew as sp_skew, kurtosis as sp_kurtosis
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

In [2]:
# Sets path
test_path = "aps_failure_test_set_processed_8bit.csv"
train_path = "aps_failure_training_set_processed_8bit.csv"

# Load the latest version
test = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/aps-failure-at-scania-trucks-data-set",
  test_path,
)

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/aps-failure-at-scania-trucks-data-set",
  train_path,
)

100%|██████████| 2.99M/2.99M [00:00<00:00, 127MB/s]

Extracting zip of aps_failure_test_set_processed_8bit.csv...


100%|██████████| 11.2M/11.2M [00:00<00:00, 68.6MB/s]

Extracting zip of aps_failure_training_set_processed_8bit.csv...


# 1. Analisis del dataset


In [3]:
# Muestra de los datos
print(df.head(5))

      class    aa_000    ab_000    ac_000    ad_000    ae_000    af_000  \
0 -0.992188  0.117188 -0.289062  0.992188 -0.007812 -0.046875 -0.054688   
1 -0.992188 -0.179688 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   
2 -0.992188 -0.125000 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   
3 -0.992188 -0.406250 -0.289062 -0.468750 -0.007812 -0.046875 -0.007812   
4 -0.992188  0.007812 -0.289062 -0.468750 -0.007812 -0.046875 -0.054688   

     ag_000   ag_001    ag_002  ...    ee_002    ee_003    ee_004    ee_005  \
0 -0.007812 -0.03125 -0.054688  ...  0.687500  0.515625  0.234375  0.070312   
1 -0.007812 -0.03125 -0.054688  ... -0.023438 -0.062500 -0.132812 -0.132812   
2 -0.007812 -0.03125 -0.054688  ... -0.140625 -0.093750 -0.015625  0.015625   
3 -0.007812 -0.03125 -0.054688  ... -0.382812 -0.382812 -0.375000 -0.351562   
4 -0.007812 -0.03125 -0.054688  ...  0.156250  0.031250 -0.031250 -0.039062   

     ee_006    ee_007    ee_008    ee_009    ef_000    eg_000  
0  0.00781

In [7]:
# Tamaño del dataset
print(df.shape)

(60000, 171)


Como que 60k filas 💀, que diga...

Descripcion de las columnas:
- clase: "neg" indica fallo en el sistema APS, "pos" todo en orden. (se renombra por palabra reservada)
- aa_000, ab_000, ac_000, ..., eg_000: atributos de entrada, sin un significado en particular (anonimos)

In [5]:
# Columnas del dataset
df.rename(columns={'class': 'clase'}, inplace=True)
df.columns

Index(['clase', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000',
       'ag_000', 'ag_001', 'ag_002',
       ...
       'ee_002', 'ee_003', 'ee_004', 'ee_005', 'ee_006', 'ee_007', 'ee_008',
       'ee_009', 'ef_000', 'eg_000'],
      dtype='object', length=171)

In [10]:
# Cantidad de elementos vacios por cada columna
cant_vacios = (df == "na").sum()
cant_vacios = cant_vacios[cant_vacios > 0]
print(cant_vacios)

Series([], dtype: int64)


In [11]:
# Cantidad de positivos y negativos del dataset
cant_neg = len(df[df.clase == 'neg'])
cant_pos = len(df[df.clase == 'pos'])
print(f"Negativos: {cant_neg}, Positivos: {cant_pos}")

Negativos: 0, Positivos: 0


In [16]:
# =========================
# 1) Librerías y parámetros
# =========================
import re
import numpy as np
import pandas as pd

CSV_PATH   = "tu_dataset.csv"   # <-- cámbialo
TARGET_COL = "class"            # cambia si tu etiqueta se llama distinto
MIN_BINS_FOR_HIST = 3           # mínimo de columnas por base para considerarlo histograma

# Abreviaturas :
ABBR = {
    "mean": "med",    # media
    "var":  "var",    # varianza
    "skew": "asi",    # asimetría
    "kurt": "cur",    # curtosis (Pearson, no-excesiva)
}

# Patrón base_### (ej.: aaa_000)
SUFFIX_RE = re.compile(r"^(?P<base>[A-Za-z]+)_(?P<bin>\d{3})$")


# ==========================
# 2) Detección de histogramas
# ==========================
def infer_histogram_groups(columns, target_col=TARGET_COL, min_bins=MIN_BINS_FOR_HIST):
    """
    Devuelve:
      - hist_groups: dict { base: [cols ordenadas por bin] }
      - non_hist_cols: columnas que no pertenecen a histogramas
    """
    buckets = {}
    non_hist_cols = []

    for col in columns:
        if col == target_col:
            continue
        m = SUFFIX_RE.match(col)
        if not m:
            non_hist_cols.append(col)
            continue
        base = m.group("base")
        bin_id = int(m.group("bin"))
        buckets.setdefault(base, []).append((bin_id, col))

    hist_groups = {}
    for base, pairs in buckets.items():
        pairs_sorted = sorted(pairs, key=lambda x: x[0])
        cols_sorted = [c for _, c in pairs_sorted]
        if len(cols_sorted) >= min_bins:
            hist_groups[base] = cols_sorted
        else:
            non_hist_cols.extend(cols_sorted)

    return hist_groups, non_hist_cols




def summarize_histogram_matrix_robust(mat, laplace_alpha=0.0, use_float=np.float64):
    """
    mat: array (n_rows x n_bins) con bins en 0..255 (u otra escala de 8 bits).
    Convierte cada fila a PMF SOLO para calcular momentos (no modifica tu DF).
    Devuelve dict con mean, var, skew, kurt (kurtosis de Pearson, no-excesiva).
    Filas sin masa válida => NaN en todos los estadísticos.
    """
    M = np.array(mat, dtype=use_float, copy=True)

    totals = np.nansum(M, axis=1, keepdims=True)
    all_nan_row = np.all(~np.isfinite(M), axis=1, keepdims=True)
    invalid = (totals <= 0) | ~np.isfinite(totals) | all_nan_row

    if laplace_alpha > 0:
        M = np.where(np.isfinite(M), M + laplace_alpha, laplace_alpha)

    totals = np.nansum(M, axis=1, keepdims=True)
    invalid = invalid | (totals <= 0) | ~np.isfinite(totals)

    totals_safe = np.where(invalid, np.nan, totals)
    p = M / totals_safe  # PMF por fila (solo para los momentos)

    n_bins = M.shape[1]
    x = np.arange(n_bins, dtype=use_float).reshape(1, -1)

    mu = np.nansum(p * x, axis=1, keepdims=True)
    xc = x - mu
    m2 = np.nansum(p * (xc**2), axis=1, keepdims=True)
    m3 = np.nansum(p * (xc**3), axis=1, keepdims=True)
    m4 = np.nansum(p * (xc**4), axis=1, keepdims=True)

    var = m2.squeeze(1)
    std = np.sqrt(var)

    with np.errstate(divide='ignore', invalid='ignore'):
        skew = (m3.squeeze(1)) / (std**3)
        kurt = (m4.squeeze(1)) / (var**2)

    mu   = mu.squeeze(1)
    skew = np.where(invalid.squeeze(1), np.nan, skew)
    kurt = np.where(invalid.squeeze(1), np.nan, kurt)

    return {"mean": mu, "var": var, "skew": skew, "kurt": kurt}



def three_letter_base(base):
    """Primeras 3 letras del base en minúsculas (o lo que haya si <3)."""
    return base[:3].lower()


# ===========================
# 4) Pipeline principal (E2E)
# ===========================


# Convertir a numérico cuando se pueda
for c in df.columns:
    if c != TARGET_COL:
        df[c] = pd.to_numeric(df[c], errors="ignore")

# 4.2 Detectar histogramas
hist_groups, non_hist_cols = infer_histogram_groups(df.columns)

print("Histogramas detectados:")
for base, cols in hist_groups.items():
    print(f"  {base}: {len(cols)} bins -> {cols[:5]}{' ...' if len(cols) > 5 else ''}")
print("\nColumnas NO histogramas (muestra):", non_hist_cols[:10])

# 4.3 Construir DataFrame de salida con:
#     - class (si existe)
#     - columnas NO histogramas (se conservan)
out_cols = []
if TARGET_COL in df.columns:
    out_cols.append(TARGET_COL)
out_cols += non_hist_cols

df_out = df[out_cols].copy()

# Añadir features por cada histograma detectado
for base, cols in hist_groups.items():
    X = df[cols].astype(np.float64).values  # bins 8-bit a float64

    stats = summarize_histogram_matrix_robust(X, laplace_alpha=1e-9)

    b3 = three_letter_base(base)
    df_out[f"{b3}_{ABBR['mean']}"] = stats["mean"]
    df_out[f"{b3}_{ABBR['var']}"]  = stats["var"]
    df_out[f"{b3}_{ABBR['skew']}"] = stats["skew"]
    df_out[f"{b3}_{ABBR['kurt']}"] = stats["kurt"]


# 4.5 Ordenar: class primero si existe
if TARGET_COL in df_out.columns:
    cols_order = [TARGET_COL] + [c for c in df_out.columns if c != TARGET_COL]
    df_out = df_out[cols_order]

print("\nForma final:", df_out.shape)
print("Primeras columnas:", df_out.columns[:12].tolist())


/tmp/ipython-input-342683746.py:120: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors="ignore")


Histogramas detectados:
  ag: 10 bins -> ['ag_000', 'ag_001', 'ag_002', 'ag_003', 'ag_004'] ...
  ay: 10 bins -> ['ay_000', 'ay_001', 'ay_002', 'ay_003', 'ay_004'] ...
  az: 10 bins -> ['az_000', 'az_001', 'az_002', 'az_003', 'az_004'] ...
  ba: 10 bins -> ['ba_000', 'ba_001', 'ba_002', 'ba_003', 'ba_004'] ...
  cn: 10 bins -> ['cn_000', 'cn_001', 'cn_002', 'cn_003', 'cn_004'] ...
  cs: 10 bins -> ['cs_000', 'cs_001', 'cs_002', 'cs_003', 'cs_004'] ...
  ee: 10 bins -> ['ee_000', 'ee_001', 'ee_002', 'ee_003', 'ee_004'] ...

Columnas NO histogramas (muestra): ['clase', 'am_0', 'ec_00', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ah_000']


/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)
/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)



Forma final: (60000, 129)
Primeras columnas: ['clase', 'am_0', 'ec_00', 'aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000', 'af_000', 'ah_000', 'ai_000', 'aj_000']


/tmp/ipython-input-342683746.py:93: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(var)


In [22]:
# --- Google Colab: procesamiento de histogramas en columnas xx_000..xx_009 ---

import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Utilidades numéricas (sin dependencias externas)
# ------------------------------------------------------------
def _safe_row_pmf(arr):
    """Convierte una fila de 10 bins en una PMF (long 10).
       Si suma <= 0 o hay no finitos => devuelve NaNs."""
    arr = np.where(np.isfinite(arr), arr, 0.0)
    arr = np.where(arr < 0, 0.0, arr)  # evita negativos
    s = arr.sum(axis=1, keepdims=True)
    pmf = np.divide(arr, s, out=np.full_like(arr, np.nan), where=(s > 0))
    return pmf

def _row_entropy(pmf, log_base=np.e):
    """Entropía de Shannon por fila (nats por defecto).
       Soporta pmf con NaNs: resultado NaN si pmf inválida."""
    # Evita log(0): definimos 0*log(0) = 0 usando máscara
    with np.errstate(divide='ignore', invalid='ignore'):
        logp = np.log(pmf) if log_base == np.e else np.log(pmf)/np.log(log_base)
        term = np.where(pmf > 0, pmf * logp, 0.0)
        H = -np.nansum(term, axis=1)
    # Si la fila tenía NaN en pmf (p.ej. suma 0), deja NaN en entropía
    all_nan_row = np.all(~np.isfinite(pmf), axis=1)
    H = np.where(all_nan_row, np.nan, H)
    return H

def _row_js_divergence_vs_uniform(pmf, log_base=np.e):
    """Jensen–Shannon divergence entre pmf y uniforme (10 bins)."""
    n = pmf.shape[1]
    U = np.full((pmf.shape[0], n), 1.0/n)
    M = 0.5 * (pmf + U)
    # KL(P||M) y KL(U||M)
    def _kl(P, Q):
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where((P > 0) & (Q > 0), P / Q, 1.0)  # 0*log(0/q)=0 ; p>0,q=0 -> inf (máscara)
            log_ratio = np.log(ratio) if log_base == np.e else np.log(ratio)/np.log(log_base)
            term = np.where((P > 0) & (Q > 0), P * log_ratio, 0.0)
            kl = np.nansum(term, axis=1)
        # filas inválidas (pmf NaN) => NaN
        invalid = np.any(~np.isfinite(P), axis=1) | np.any(~np.isfinite(Q), axis=1)
        kl = np.where(invalid, np.nan, kl)
        return kl
    js = 0.5 * _kl(pmf, M) + 0.5 * _kl(U, M)
    return js

def _row_weighted_stats_on_indices(pmf):
    """Media, varianza, std sobre índices 0..9 usando pmf (para 10 bins).
       Devuelve (mean_idx, std_idx). Si pmf inválida => (NaN, NaN)."""
    n = pmf.shape[1]
    idx = np.arange(n, dtype=float)
    # media
    mean_idx = np.nansum(pmf * idx, axis=1)
    # var = E[X^2] - (E[X])^2
    ex2 = np.nansum(pmf * (idx**2), axis=1)
    var = ex2 - mean_idx**2
    var = np.where(var < 0, 0.0, var)  # estabilidad numérica
    std_idx = np.sqrt(var)
    # filas inválidas
    invalid = np.any(~np.isfinite(pmf), axis=1)
    mean_idx = np.where(invalid, np.nan, mean_idx)
    std_idx  = np.where(invalid, np.nan, std_idx)
    return mean_idx, std_idx

def _row_discrete_median_index(pmf):
    """Mediana discreta (índice 0..9) como cuantil 0.5 de la pmf."""
    n_rows, n_bins = pmf.shape
    cdf = np.nancumsum(pmf, axis=1)
    # Índice mínimo donde CDF >= 0.5
    med_idx = np.full(n_rows, np.nan)
    for i in range(n_rows):
        if np.any(~np.isfinite(pmf[i])):  # pmf inválida
            med_idx[i] = np.nan
            continue
        j = np.searchsorted(cdf[i], 0.5, side='left')
        if j >= n_bins:  # protección
            j = n_bins - 1
        med_idx[i] = float(j)
    return med_idx

def _row_right_tail_mass(pmf, tail_bins=(7,8,9)):
    """Suma de probabilidad en los bins de cola derecha (por defecto 7-9)."""
    mask = np.zeros(pmf.shape[1], dtype=bool)
    mask[list(tail_bins)] = True
    tail = np.nansum(pmf[:, mask], axis=1)
    invalid = np.any(~np.isfinite(pmf), axis=1)
    tail = np.where(invalid, np.nan, tail)
    return tail

# ------------------------------------------------------------
# Detección de grupos histograma y transformación del DataFrame
# ------------------------------------------------------------
HIST_RE = re.compile(r'^([a-z]{2})_(\d{3})$')  # ej: ea_000

def find_histogram_groups(columns):
    """
    Devuelve dict prefix -> lista ordenada de 10 columnas [xx_000..xx_009]
    Solo considera grupos donde EXISTEN exactamente los 10 bins 000..009.
    """
    by_prefix = {}
    # 1) indexar columnas por prefijo y lista de sufijos disponibles
    for col in columns:
        m = HIST_RE.match(col)
        if not m:
            continue
        pfx, suf = m.group(1), m.group(2)
        by_prefix.setdefault(pfx, set()).add(suf)

    # 2) filtrar los que tienen 000..009
    groups = {}
    need = {f"{i:03d}" for i in range(10)}
    for pfx, sufs in by_prefix.items():
        if need.issubset(sufs):
            ordered_cols = [f"{pfx}_{i:03d}" for i in range(10)]
            groups[pfx] = ordered_cols
    return groups

def transform_histograms(df, entropy_base=np.e, right_tail_bins=(7,8,9), drop_original=True):
    """
    Para cada grupo histograma (xx_000..xx_009):
      - Calcula: med (mediana discreta), des (std de índices), ent (Shannon),
                 col (cola derecha), div (JS divergence vs uniforme)
      - Inserta columnas xx_med, xx_des, xx_ent, xx_col, xx_div
      - Borra columnas originales del histograma si drop_original=True
    """
    df = df.copy()
    groups = find_histogram_groups(df.columns)

    for pfx, bin_cols in groups.items():
        # Extrae matriz (n_filas x 10)
        mat = df[bin_cols].to_numpy(dtype=float, copy=False)
        pmf = _safe_row_pmf(mat)

        # Entropía y JS divergence (mismo log_base)
        ent = _row_entropy(pmf, log_base=entropy_base)
        jsd = _row_js_divergence_vs_uniform(pmf, log_base=entropy_base)

        # Estadísticos sobre índices 0..9
        mean_idx, std_idx = _row_weighted_stats_on_indices(pmf)
        median_idx = _row_discrete_median_index(pmf)

        # Cola derecha
        right_tail = _row_right_tail_mass(pmf, tail_bins=right_tail_bins)

        # Nombres de salida siguiendo tu convención
        out_cols = [
            f"{pfx}_med",  # mediana
            f"{pfx}_des",  # std
            f"{pfx}_ent",  # entropía
            f"{pfx}_col",  # cola derecha
            f"{pfx}_div",  # JS divergence
        ]
        out_data = np.column_stack([median_idx, std_idx, ent, right_tail, jsd])

        # Insertar a la derecha del último bin del grupo
        insert_after_col = bin_cols[-1]
        insert_pos = list(df.columns).index(insert_after_col) + 1
        for j, colname in enumerate(out_cols):
            df.insert(insert_pos + j, colname, out_data[:, j])

        # Eliminar los 10 bins originales (si aplica)
        if drop_original:
            df.drop(columns=bin_cols, inplace=True)

    return df

# ------------------------------------------------------------
# EJEMPLO DE USO
# ------------------------------------------------------------
# Supón que ya cargaste tu dataset:
# df = pd.read_csv('/content/tu_archivo.csv')
# df_tr = transform_histograms(df, entropy_base=np.e, right_tail_bins=(7,8,9), drop_original=True)
# df_tr.head()


In [26]:
df_out = fit_transform_pipeline(df)
df_out.head()

,clase,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_med,ag_des,ag_ent,ag_col,ag_div,ah_000,ai_000,aj_000,ak_000,al_000,am_0,an_000,ao_000,ap_000,aq_000,ar_000,as_000,at_000,au_000,av_000,ax_000,ay_med,ay_des,ay_ent,ay_col,ay_div,az_med,az_des,az_ent,az_col,az_div,ba_med,ba_des,ba_ent,ba_col,ba_div,bb_000,bc_000,bd_000,be_000,bf_000,bg_000,bh_000,bi_000,bj_000,bk_000,bl_000,bm_000,bn_000,bo_000,bp_000,bq_000,br_000,bs_000,bt_000,bu_000,bv_000,bx_000,by_000,bz_000,ca_000,cb_000,cc_000,cd_000,ce_000,cf_000,cg_000,ch_000,ci_000,cj_000,ck_000,cl_000,cm_000,cn_med,cn_des,cn_ent,cn_col,cn_div,co_000,cp_000,cq_000,cr_000,cs_med,cs_des,cs_ent,cs_col,cs_div,ct_000,cu_000,cv_000,cx_000,cy_000,cz_000,da_000,db_000,dc_000,dd_000,de_000,df_000,dg_000,dh_000,di_000,dj_000,dk_000,dl_000,dm_000,dn_000,do_000,dp_000,dq_000,dr_000,ds_000,dt_000,du_000,dv_000,dx_000,dy_000,dz_000,ea_000,eb_000,ec_00,ed_000,ee_med,ee_des,ee_ent,ee_col,ee_div,ef_000,eg_000
0,-0.992188,0.559055,0.354331,1.00000,0.496063,0.476378,0.472441,0.555556,0.624848,0.991995,0.312126,0.006447,0.586614,0.472441,0.488189,0.488189,0.444882,0.444882,0.594488,0.547244,0.622047,0.771654,0.448819,0.496063,0.480315,0.492126,0.488189,0.460630,0.555556,0.608661,0.984761,0.253450,0.011724,0.555556,0.630083,0.992867,0.292468,0.005596,0.555556,0.591991,0.973556,0.201458,0.021340,0.598425,0.421260,0.401575,0.425197,0.472441,0.586614,0.629921,0.653543,0.578740,0.377953,0.393701,0.358268,0.374016,0.358268,0.090551,0.098425,0.098425,1.000000,0.559055,0.598425,0.598425,0.618110,0.700787,0.464567,0.700787,0.814961,0.622047,0.5,1.000000,0.496063,0.492126,0.492126,0.606299,0.452756,0.547244,0.444882,0.937008,0.555556,0.601598,0.984608,0.276136,0.012229,0.496063,0.476378,0.598425,0.456693,0.444444,0.639644,0.995926,0.270620,0.003344,0.472441,0.456693,0.720472,0.393701,0.476378,1.000000,0.480315,0.500000,0.783465,0.566929,0.712598,0.488189,0.484252,0.496063,0.448819,0.496063,0.484252,0.488189,0.484252,0.645669,0.913386,1.000000,0.472441,0.417323,0.724409,0.779528,0.929134,0.775591,0.397638,0.429134,0.488189,0.484252,0.405512,0.622047,0.649606,0.444444,0.604643,0.987850,0.222752,0.009981,0.488189,0.488189
1,-0.992188,0.409449,0.354331,0.26378,0.496063,0.476378,0.472441,0.555556,0.648848,0.998953,0.314167,0.000878,0.448819,0.472441,0.488189,0.488189,0.444882,0.444882,0.440945,0.437008,0.511811,0.456693,0.448819,0.496063,0.480315,0.492126,0.413386,0.370079,0.444444,0.636386,0.998547,0.296053,0.001229,0.444444,0.647470,0.998995,0.297231,0.000829,0.555556,0.645796,0.996534,0.341730,0.002825,0.460630,0.433071,0.444882,0.440945,0.712598,0.448819,0.468504,0.566929,0.468504,0.389764,0.397638,0.425197,0.078740,0.086614,0.090551,0.098425,0.098425,0.062992,0.409449,0.460630,0.460630,0.472441,0.460630,0.606299,1.000000,0.000000,0.452756,0.5,0.275591,0.496063,0.433071,0.492126,0.429134,0.452756,0.484252,0.444882,0.389764,0.555556,0.641599,0.999232,0.304940,0.000645,0.496063,0.460630,0.460630,0.531496,0.444444,0.652089,0.998833,0.304455,0.000979,0.543307,0.503937,0.409449,0.448819,0.480315,0.468504,0.480315,0.425197,0.484252,0.496063,0.535433,0.488189,0.484252,0.496063,0.448819,0.496063,0.484252,0.488189,0.484252,0.500000,0.551181,0.456693,0.472441,0.417323,0.511811,0.492126,0.507874,0.551181,0.397638,0.429134,0.488189,0.484252,0.413386,0.590551,0.598425,0.444444,0.644377,0.998961,0.289314,0.000861,0.488189,0.488189
2,-0.992188,0.437008,0.354331,0.26378,0.496063,0.476378,0.472441,0.444444,0.650616,0.998201,0.296647,0.001498,0.429134,0.472441,0.488189,0.488189,0.444882,0.444882,0.429134,0.437008,0.385827,0.385827,0.448819,0.496063,0.480315,0.492126,0.440945,0.405512,0.444444,0.645502,0.998216,0.295960,0.001512,0.444444,0.641254,0.998717,0.309060,0.001069,0.444444,0.631536,0.999148,0.285588,0.000707,0.413386,0.437008,0.421260,0.448819,0.437008,0.429134,0.405512,0.385827,0.397638,0.165354,0.157480,0.437008,0.444882,0.448819,0.452756,0.456693,0.456693,0.188976,0.437008,0.413386,0.413386,0.429134,0.437008,0.421260,0.26

In [24]:
# ============================================
# Colab: Escalado global [0,1] + features de histogramas normalizadas a [0,1]
# ============================================
import re, json, math
import numpy as np
import pandas as pd

# ---------- Parámetros de escala global conocidos ----------
GLOBAL_MIN = -0.9921875
GLOBAL_MAX =  0.9921875
GLOBAL_RANGE = GLOBAL_MAX - GLOBAL_MIN

# ---------- Regex de histogramas ----------
HIST_RE = re.compile(r'^([a-z]{2})_(\d{3})$')  # ej: ea_000..ea_009

# ---------- Utilidades ----------
def global_minmax_scale(df, cols):
    """Escala con la MISMA escala global [-0.9921875,0.9921875] -> [0,1]."""
    out = df.copy()
    out[cols] = (out[cols] - GLOBAL_MIN) / GLOBAL_RANGE
    out[cols] = out[cols].clip(lower=0.0, upper=1.0)
    return out

def find_histogram_groups(columns):
    """Devuelve dict prefix -> [xx_000..xx_009] si el grupo está completo."""
    by_prefix = {}
    for col in columns:
        m = HIST_RE.match(col)
        if not m:
            continue
        pfx, suf = m.group(1), m.group(2)
        by_prefix.setdefault(pfx, set()).add(suf)
    need = {f"{i:03d}" for i in range(10)}
    groups = {}
    for pfx, sufs in by_prefix.items():
        if need.issubset(sufs):
            groups[pfx] = [f"{pfx}_{i:03d}" for i in range(10)]
    return groups

def row_pmf_from_bins(arr):
    """PMF fila a fila (ya todo >=0 por el escalado)."""
    arr = np.where(np.isfinite(arr), arr, 0.0)
    s = arr.sum(axis=1, keepdims=True)
    # si s==0 -> uniforme para evitar NaNs
    zero = (s <= 0)
    if np.any(zero):
        n = arr.shape[1]
        arr = arr.copy()
        arr[zero[:,0], :] = 1.0 / n
        s = arr.sum(axis=1, keepdims=True)
    pmf = arr / s
    return pmf

def row_entropy_bits(pmf):
    """Entropía de Shannon (bits)."""
    with np.errstate(divide='ignore', invalid='ignore'):
        log2p = np.log2(pmf)
        term = np.where(pmf > 0, pmf * log2p, 0.0)
        H = -np.nansum(term, axis=1)
    return H

def row_js_divergence_base2(pmf):
    """Jensen–Shannon divergence con log base 2 (acotada por 1)."""
    n = pmf.shape[1]
    U = np.full((pmf.shape[0], n), 1.0/n)
    M = 0.5*(pmf + U)
    def _kl2(P, Q):
        with np.errstate(divide='ignore', invalid='ignore'):
            mask = (P > 0) & (Q > 0)
            ratio = np.where(mask, P/Q, 1.0)
            return np.nansum(np.where(mask, P*np.log2(ratio), 0.0), axis=1)
    return 0.5*_kl2(pmf, M) + 0.5*_kl2(U, M)

def row_discrete_median_index(pmf):
    cdf = np.cumsum(pmf, axis=1)
    j = (cdf >= 0.5).argmax(axis=1)  # primer índice donde cdf >= 0.5
    return j.astype(float)

def row_weighted_std_on_indices(pmf):
    idx = np.arange(pmf.shape[1], dtype=float)
    mean = np.sum(pmf*idx, axis=1)
    ex2  = np.sum(pmf*(idx**2), axis=1)
    var  = np.maximum(ex2 - mean**2, 0.0)
    return np.sqrt(var)

def row_right_tail_mass(pmf, tail=(7,8,9)):
    mask = np.zeros(pmf.shape[1], dtype=bool)
    mask[list(tail)] = True
    return np.sum(pmf[:, mask], axis=1)

def transform_histograms_and_normalize(df, drop_original=True):
    """
    - Detecta grupos xx_000..xx_009
    - Calcula med, des, ent(bits), col, div(base2)
    - Normaliza esas 5 features a [0,1] con límites teóricos
    - Inserta tras el último bin y (opcional) elimina los bins
    """
    df = df.copy()
    groups = find_histogram_groups(df.columns)
    LOG2_10 = math.log2(10.0)
    for pfx, cols in groups.items():
        mat = df[cols].to_numpy(dtype=float, copy=False)
        pmf = row_pmf_from_bins(mat)

        med = row_discrete_median_index(pmf) / 9.0            # [0,1]
        des = row_weighted_std_on_indices(pmf) / 4.5           # [0,1]
        ent = row_entropy_bits(pmf) / LOG2_10                  # [0,1]
        col = row_right_tail_mass(pmf, (7,8,9))                # [0,1]
        div = row_js_divergence_base2(pmf)                     # ≤1
        div = np.clip(div, 0.0, 1.0)

        out_cols = [f"{pfx}_med", f"{pfx}_des", f"{pfx}_ent", f"{pfx}_col", f"{pfx}_div"]
        out_data = np.column_stack([med, des, ent, col, div])

        insert_pos = list(df.columns).index(cols[-1]) + 1
        for j, c in enumerate(out_cols):
            df.insert(insert_pos + j, c, out_data[:, j])

        if drop_original:
            df.drop(columns=cols, inplace=True)

    return df, list(groups.keys())

# ---------- Ejemplo de uso ----------
# df = pd.read_csv('/content/tu_archivo.csv')
# Si tienes una columna de etiqueta, exclúyela del escalado global:
label_col = 'clase'  # cámbialo si tu etiqueta se llama distinto

def fit_transform_pipeline(df):
    df = df.copy()
    # 1) columnas numéricas a escalar globalmente (excluye etiqueta si existe)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if label_col in num_cols:
        num_cols.remove(label_col)

    # 2) Escalado global uniforme [0,1] (MISMA escala para todo el dataset)
    df_s = global_minmax_scale(df, num_cols)

    # 3) Transformación de histogramas + normalización a [0,1]
    df_t, prefixes = transform_histograms_and_normalize(df_s, drop_original=True)

    # 4) Guardar la "escala" para reproducirla luego en datos nuevos
    scaler = {
        "version": 1,
        "global_min": GLOBAL_MIN,
        "global_max": GLOBAL_MAX,
        "global_range": GLOBAL_RANGE,
        "scaled_columns": num_cols,          # orden actual
        "label_column": label_col,
        "hist_prefixes": prefixes,
        "norm_constants": {
            "median_div": 9.0,
            "std_div": 4.5,
            "entropy_div": math.log2(10.0),
            "jsd_div": 1.0,
            "right_tail_bins": [7,8,9]
        }
    }
    with open('/content/scaler_hist.json', 'w') as f:
        json.dump(scaler, f, indent=2)

    return df_t

# df_out = fit_transform_pipeline(df)
# df_out.to_csv('/content/dataset_transformado.csv', index=False)
# ¡Listo! El JSON queda en /content/scaler_hist.json


In [17]:
pd.set_option('display.max_columns', None)  # no cortar columnas
print(df_out.head())

      clase      am_0     ec_00    aa_000    ab_000    ac_000    ad_000  \
0 -0.992188 -0.109375  0.242188  0.117188 -0.289062  0.992188 -0.007812   
1 -0.992188 -0.109375  0.179688 -0.179688 -0.289062 -0.468750 -0.007812   
2 -0.992188 -0.109375 -0.132812 -0.125000 -0.289062 -0.468750 -0.007812   
3 -0.992188 -0.101562 -0.406250 -0.406250 -0.289062 -0.468750 -0.007812   
4 -0.992188 -0.109375 -0.109375  0.007812 -0.289062 -0.468750 -0.007812   

     ae_000    af_000    ah_000    ai_000    aj_000    ak_000    al_000  \
0 -0.046875 -0.054688  0.171875 -0.054688 -0.023438 -0.023438 -0.109375   
1 -0.046875 -0.054688 -0.101562 -0.054688 -0.023438 -0.023438 -0.109375   
2 -0.046875 -0.054688 -0.140625 -0.054688 -0.023438 -0.023438 -0.109375   
3 -0.046875 -0.007812 -0.429688 -0.054688 -0.023438 -0.023438 -0.109375   
4 -0.046875 -0.054688  0.039062 -0.054688 -0.015625 -0.023438 -0.109375   

     an_000    ao_000    ap_000    aq_000    ar_000    as_000    at_000  \
0  0.187500  0.093750  

In [21]:
import numpy as np
import pandas as pd

def audit_hist_group(df_out, cols, name="hist", top_n=5):
    X = df_out[cols].astype(float).values
    totals = np.nansum(X, axis=1)
    pos_bins = (X > 0).sum(axis=1)
    # var usando índices de bin como soporte, con PMF
    n_bins = X.shape[1]
    x = np.arange(n_bins).reshape(1, -1)
    totals_safe = np.where(totals==0, np.nan, totals)[:, None]
    p = X / totals_safe
    mu = np.nansum(p * x, axis=1)
    var = np.nansum(p * (x - mu[:, None])**2, axis=1)

    print(f"\n[AUDIT {name}] filas: {len(df)}")
    print(" total==0:", int((totals==0).sum()))
    print(" pos_bins==1 (toda la masa en un bin):", int((pos_bins==1).sum()))
    print(" var==0   :", int((var==0).sum()))
    # casos sospechosos: var==0 pero hay >1 bin positivo
    mask_weird = (var==0) & (pos_bins>1) & (totals>0)
    idxs = np.where(mask_weird)[0][:top_n]
    if len(idxs):
        print(f" casos sospechosos (var==0 & >1 bin>0): {len(idxs)} (mostrando {top_n}) ->", idxs.tolist())
        display(df.iloc[idxs][cols])
    else:
        print(" sin casos sospechosos (bien).")

# ejemplo: auditar todos los histogramas encontrados
for base, cols in hist_groups.items():
    audit_hist_group(df, cols, name=base)


[AUDIT ag] filas: 60000
 total==0: 66
 pos_bins==1 (toda la masa en un bin): 7818
 var==0   : 66
 sin casos sospechosos (bien).

[AUDIT ay] filas: 60000
 total==0: 90
 pos_bins==1 (toda la masa en un bin): 14252
 var==0   : 90
 sin casos sospechosos (bien).

[AUDIT az] filas: 60000
 total==0: 70
 pos_bins==1 (toda la masa en un bin): 13814
 var==0   : 70
 sin casos sospechosos (bien).

[AUDIT ba] filas: 60000
 total==0: 34
 pos_bins==1 (toda la masa en un bin): 9160
 var==0   : 34
 sin casos sospechosos (bien).

[AUDIT cn] filas: 60000
 total==0: 56
 pos_bins==1 (toda la masa en un bin): 7483
 var==0   : 56
 sin casos sospechosos (bien).

[AUDIT cs] filas: 60000
 total==0: 62
 pos_bins==1 (toda la masa en un bin): 6490
 var==0   : 62
 sin casos sospechosos (bien).

[AUDIT ee] filas: 60000
 total==0: 45
 pos_bins==1 (toda la masa en un bin): 6062
 var==0   : 45
 sin casos sospechosos (bien).


# 2. Preprocesamiento del dataset